In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gsd.hoomd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


In [ ]:
GSD_DIR   = Path("/home/gabriel/Projects/Anchoring_Study/anch_frac_0.5/Results/lc_length_6/GSD")
CACHE_DIR = Path("/home/gabriel/Projects/Anchoring_Study/anch_frac_0.5/Results/lc_length_6/cache")
CACHE_DIR.mkdir(exist_ok=True)

PHASES   = ["nematic", "smectic"]
LC_FRACS = [0.3, 0.5, 0.7]
PHASE_LABELS = {
    "nematic": r"Nematic ($\tilde{\mu}=25$)",
    "smectic": r"Smectic ($\tilde{\mu}=50$)",
}

# GSD frame indices (0-indexed); frame 0 has garbage step, frames 1-35 are usable.
# Index 13 -> step ~18K (~91 tau), 23 -> ~214K (~1068 tau), 33 -> ~1.6M (~7984 tau)
FRAME_INDICES = [13, 23, 33]
FRAME_LABELS  = {
    13: r"$t \approx 91\,\tau$",
    23: r"$t \approx 1068\,\tau$",
    33: r"$t \approx 7984\,\tau$",
}

NX      = 60    # grid points per axis (60x60x60)
LC_TYPES = [2, 3]  # GSD typeids for both halves of each LC molecule
# False = center pair (beads 3,4) — exactly matches MATILDA's lc.input assignment
# True  = end-to-end vector (beads 0,5) — more noise-robust for stiff chains
USE_END_TO_END = False

THRESH     = 0.05   # exclude near-zero polymer-rich voxels
CMAP       = "hot"  # dark=low order, bright=high order
VMAX       = 0.7
ELEV, AZIM = 25, 45


In [ ]:
def compute_op_from_gsd(gsd_path, frame_idx=-1):
    """Compute density-normalized nematic order parameter on the 60^3 grid.

    Replicates MATILDA's DistributeSTensors (center-pair S broadcast to all
    6 beads, CIC-mapped to grid) but then divides the accumulated S_field at
    each voxel by the scalar density weight rho_field before computing the
    max eigenvalue.  This gives the TRUE local S in [0, 1], independent of
    how many molecules happen to sit near a given voxel — necessary for
    phase-separated systems where the LC-rich domain has ~3x average density.
    (MATILDA's un-normalized output happens to fall in [0,1] only for
    uniform-density pure-LC simulations with ~1 bead/voxel.)
    """
    with gsd.hoomd.open(str(gsd_path), 'r') as f:
        frame = f[frame_idx]

    pos = frame.particles.position           # (N, 3), in [-L/2, L/2]
    tp  = frame.particles.typeid             # (N,)
    mol = frame.log['particles/moleculeid']  # (N,)
    L   = float(frame.configuration.box[0])
    Lh  = L / 2.0

    # 1. Select LC particles, sort by (mol_id, global_idx) -> (N_mol, 6, 3)
    lc_idx = np.where(np.isin(tp, LC_TYPES))[0]
    lc_mol = mol[lc_idx]
    order  = np.lexsort((lc_idx, lc_mol))
    lc_idx_sorted = lc_idx[order]
    N_mol   = len(np.unique(lc_mol))
    pos_mat = pos[lc_idx_sorted].reshape(N_mol, 6, 3)

    # 2. Orientation vector: center pair (beads 3,4) or end-to-end (beads 0,5)
    if USE_END_TO_END:
        r1, r2 = pos_mat[:, 0, :], pos_mat[:, 5, :]
    else:
        r1, r2 = pos_mat[:, 3, :], pos_mat[:, 4, :]

    dr  = r2 - r1
    dr -= L * np.round(dr / L)
    mag = np.linalg.norm(dr, axis=1, keepdims=True)
    u   = dr / mag

    # 3. S-tensor per molecule: u x u - I/3, shape (N_mol, 9)
    S_mol = (u[:, :, None] * u[:, None, :]).reshape(N_mol, 9)
    S_mol[:, [0, 4, 8]] -= 1.0 / 3.0

    # 4. Broadcast molecule S to all 6 beads -> (N_mol*6, 9)
    S_bead  = np.repeat(S_mol, 6, axis=0)
    all_pos = pos_mat.reshape(-1, 3)

    # 5. CIC: accumulate tensor field AND scalar density weight
    g  = (all_pos + Lh) / L * NX
    i0 = np.floor(g).astype(int) % NX
    f  = g - np.floor(g)

    S_field   = np.zeros((NX**3, 9), dtype=np.float64)
    rho_field = np.zeros(NX**3,      dtype=np.float64)
    for dz, wz in [(0, 1.0 - f[:, 2]), (1, f[:, 2])]:
        for dy, wy in [(0, 1.0 - f[:, 1]), (1, f[:, 1])]:
            for dx, wx in [(0, 1.0 - f[:, 0]), (1, f[:, 0])]:
                iz   = (i0[:, 2] + dz) % NX
                iy   = (i0[:, 1] + dy) % NX
                ix   = (i0[:, 0] + dx) % NX
                flat = iz * NX * NX + iy * NX + ix
                w    = wz * wy * wx
                np.add.at(S_field,   flat, w[:, None] * S_bead)
                np.add.at(rho_field, flat, w)

    # 6. Density-normalize: divide each voxel's S_field by its accumulated weight
    #    so that the eigenvalue reflects pure orientational order, not particle count.
    mask = rho_field > 0
    S_field[mask] /= rho_field[mask, None]

    # 7. Max eigenvalue per voxel, normalize by (D-1)/D = 2/3
    vals = np.linalg.eigvalsh(S_field.reshape(-1, 3, 3))
    lam  = vals[:, 2] / (2.0 / 3.0)
    lam[~mask] = 0.0  # no LC molecules -> zero order

    iz_idx, iy_idx, ix_idx = np.unravel_index(np.arange(NX**3), (NX, NX, NX))
    return pd.DataFrame({'x': ix_idx, 'y': iy_idx, 'z': iz_idx, 'lambda': lam})


In [ ]:
data = {}  # keyed by (phase, frac, frame_idx)

for phase in PHASES:
    for frac in LC_FRACS:
        gsd_path = (GSD_DIR /
                    f"phase_separation_{phase}_lc_frac_{frac}"
                    f"_anch_str_0.0_Nb_6_rep_1_Nb_6.gsd")
        for fidx in FRAME_INDICES:
            cache_file = CACHE_DIR / f"{phase}_lc_frac_{frac}_frame{fidx}_op_norm.npy"
            if cache_file.exists():
                lam = np.load(cache_file)
                print(f"Loaded cache: {cache_file.name}")
            else:
                print(f"Computing {phase} lc_frac={frac} frame={fidx} ...", end=" ", flush=True)
                df_full = compute_op_from_gsd(gsd_path, frame_idx=fidx)
                lam = df_full["lambda"].values.astype(np.float32)
                np.save(cache_file, lam)
                print("done")
            iz, iy, ix = np.unravel_index(np.arange(NX**3), (NX, NX, NX))
            df = pd.DataFrame({'x': ix, 'y': iy, 'z': iz, 'lambda': lam})
            data[(phase, frac, fidx)] = df[df["lambda"] > THRESH]
            n = len(data[(phase, frac, fidx)])
            frac_above = 100 * n / NX**3
            print(f"  {phase} lc_frac={frac} frame={fidx}: {n:,} voxels above threshold ({frac_above:.1f}%)")


In [ ]:
for fidx in FRAME_INDICES:
    fig = plt.figure(figsize=(15, 10))
    axes = {}
    sc_last = None

    for row, phase in enumerate(PHASES):
        for col, frac in enumerate(LC_FRACS):
            ax = fig.add_subplot(2, 3, row * 3 + col + 1, projection='3d')
            axes[(phase, frac)] = ax

            sub = data[(phase, frac, fidx)]
            sc = ax.scatter(
                sub["x"], sub["y"], sub["z"],
                c=sub["lambda"],
                cmap=CMAP, vmin=THRESH, vmax=VMAX,
                s=1, alpha=0.4, linewidths=0,
            )
            sc_last = sc

            ax.set_xlim(0, 59); ax.set_ylim(0, 59); ax.set_zlim(0, 59)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
            ax.view_init(elev=ELEV, azim=AZIM)

            if row == 0:
                ax.set_title(f"$f_{{\\rm LC}}={frac}$", fontsize=13, pad=6)
            if col == 0:
                ax.text2D(-0.12, 0.5, PHASE_LABELS[phase], transform=ax.transAxes,
                          fontsize=12, va='center', rotation=90)

    cbar = fig.colorbar(sc_last, ax=list(axes.values()), shrink=0.6, pad=0.02, aspect=30)
    cbar.set_label("\u03bb (Nematic Order Parameter)", fontsize=12)

    time_label = FRAME_LABELS[fidx]
    plt.suptitle(
        f"Nematic Order Parameter — Spatial Distribution ({time_label}, anch\_str = 0.0)",
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(f"lc_order_parameter_3d_frame{fidx}.png", dpi=150, bbox_inches="tight")
    plt.show()
